<a id="top"></a>

# Demo: trace the round-trip, message by message

```yaml
title: "Demo: trace the round-trip, message by message"
keywords: round-trip, four messages, conversation history, tool_call_id, stateless, token cost, trace, langchain, ToolMessage, teacher demo
estimated duration: 10
```

> **Lesson:** L07. Teacher demo — Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). **No API key needed** — this demo dissects a *crafted transcript* of the exact exchange [Demo 2](L0704_lecture.ipynb) produces, built from real LangChain message objects, so the structure is identical every run. (In class you can instead replay the real `messages` list Demo 2 built.)
>
> The point to land: a single tool-using exchange is **at minimum four messages** — `HumanMessage → AIMessage(tool call) → ToolMessage → AIMessage` — and the call id is the thread tying request to result. Every tool call grows the history — that is the protocol, not a side effect.

## Contents

- [1. Setup — a captured four-message transcript](#1-setup--a-captured-four-message-transcript)
- [2. Walk the four messages in order](#2-walk-the-four-messages-in-order)
- [3. The id ties the result to the request](#3-the-id-ties-the-result-to-the-request)
- [4. Count the growth](#4-count-the-growth)
- [5. Takeaways](#5-takeaways)

## 1. Setup — a captured four-message transcript

The same shape Demo 2 produced, written out as real LangChain message objects so we can walk it slowly with no live call. These are the exact classes the agent loop builds: `HumanMessage`, `AIMessage` (whose `.tool_calls` carries the request), and `ToolMessage` (the result).

In [1]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.messages.tool import ToolCall

# A captured transcript of one successful round-trip (Demo 2's exchange), built
# from the same message classes the loop uses.
CALL_ID = "toolu_01ABC"  # the unique id the model assigned to its tool call

transcript: list[BaseMessage] = [
    HumanMessage(content="What is 18,374 multiplied by 92,431?"),
    AIMessage(
        content="",
        tool_calls=[
            ToolCall(
                id=CALL_ID,
                name="calculator",
                args={"expression": "18374 * 92431"},
                type="tool_call",
            )
        ],
    ),
    ToolMessage(content="1698367394", tool_call_id=CALL_ID),
    AIMessage(content="18,374 x 92,431 = 1,698,367,394."),
]
print(f"{len(transcript)} messages captured")

4 messages captured


[↑ Back to top](#top)

## 2. Walk the four messages in order

Print each message with its type, what it carries, and **who produced it**. The user 'asked once' — yet the history is four messages long.

In [2]:
def summarize(msg: BaseMessage) -> str:
    """One-line summary of a message: its class and what it carries."""
    if isinstance(msg, AIMessage) and msg.tool_calls:
        names = [c["name"] for c in msg.tool_calls]
        return f"AIMessage    tool call(s) -> {names}"
    if isinstance(msg, ToolMessage):
        return f"ToolMessage  result for id={msg.tool_call_id}"
    return f"{type(msg).__name__:11}  text -> {msg.text!r}"


producers = ["the human", "the model", "the APPLICATION", "the model"]
for i, (msg, who) in enumerate(zip(transcript, producers, strict=True), start=1):
    print(f"Message {i} | {summarize(msg)}")
    print(f"           produced by {who}")

Message 1 | HumanMessage  text -> 'What is 18,374 multiplied by 92,431?'
           produced by the human
Message 2 | AIMessage    tool call(s) -> ['calculator']
           produced by the model
Message 3 | ToolMessage  result for id=toolu_01ABC
           produced by the APPLICATION
Message 4 | AIMessage    text -> '18,374 x 92,431 = 1,698,367,394.'
           produced by the model


[↑ Back to top](#top)

## 3. The id ties the result to the request

The `ToolMessage` in Message 3 names the **same** id (`tool_call_id`) as the tool call in Message 2. That id is how the application says 'this result answers *that* request' — essential once more than one call is in flight ([L10](L0702_lecture.md)).

In [3]:
call = transcript[1].tool_calls[0]  # the tool call in message 2
result_msg = transcript[2]  # the ToolMessage in message 3
assert isinstance(result_msg, ToolMessage)
print("tool call id     (msg 2):", call["id"])
print("ToolMessage id   (msg 3):", result_msg.tool_call_id)
assert call["id"] == result_msg.tool_call_id
print("ids match -> this result answers that exact request")

tool call id     (msg 2): toolu_01ABC
ToolMessage id   (msg 3): toolu_01ABC
ids match -> this result answers that exact request


[↑ Back to top](#top)

## 4. Count the growth

The user asked **once**, but the conversation grew by **four** messages. Every future tool call repeats this growth — and the tool *definition* is re-sent on every request because the model is **stateless**.

In [4]:
print("user questions asked     :", 1)
print("messages now in history  :", len(transcript))
print()
# Tools cost tokens TWICE OVER: the definition rides in every request, and the
# result lives in the history for every later turn. A 500-token definition across
# a 10-turn chat is ~5,000 input tokens before the tool is even called.
DEFINITION_TOKENS = 500
TURNS = 10
print(f"~{DEFINITION_TOKENS * TURNS} input tokens spent on the definition alone over {TURNS} turns")

user questions asked     : 1
messages now in history  : 4

~5000 input tokens spent on the definition alone over 10 turns


[↑ Back to top](#top)

## 5. Takeaways

- A successful round-trip is **four messages**: `HumanMessage -> AIMessage(tool call) -> ToolMessage -> AIMessage(final)`. Four is the *minimum*, not a fixed number.
- The `ToolMessage` is its **own message role** — not folded into an assistant or human turn. Its `tool_call_id` links it back to the request. Role marks **protocol position**, not who is human.
- The model is **stateless across calls** — Message 4 required re-sending Messages 1-3 *and* the tool definition. The model did not 'remember' the tool.
- Tools cost tokens **twice over** — definition re-sent every request, result kept in history. Forward-link to [L12](L0702_lecture.md) (model power) and L16 (context management).

[↑ Back to top](#top)